## Hydroscope Causality Analysis Tutorial
This notebook demonstrates how to perform **causality and sensitivity analysis** using the **Hydroscope** library.  
We apply the following methods to simulation data:

- Convergent Cross Mapping (CCM)  
- Granger Causality  
- Mutual Information (MI)  
- Transfer Entropy (TE)  
- SHAP (SHapley Additive exPlanations)


## Imports
Import all required Python packages and Hydroscope methods.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm

# Hydroscope imports
import hydroscope
import hydroscope.methods.te as te
import hydroscope.methods.gc as gc
import hydroscope.methods.mi as mi
import hydroscope.methods.ccm as ccm
import hydroscope.methods.sb as sb
from hydroscope.models.testmodel1 import model
from hydroscope.methods import lagselector

ModuleNotFoundError: No module named 'hydroscope'

## Data Preprocessing
Load the hydrological dataset, convert units (e.g., streamflow), and estimate PET (Potential Evapotranspiration) using a temperature-based formula.


In [ ]:
data = pd.read_csv('lmich_data.csv', index_col='Date')
data.index = pd.to_datetime(data.index)

# Convert Streamflow (SF) from cfs to mm/day
data['SF'] = 0.028316847 * data['SF'] * 86400 * 1e3 / (149 * 2.59e6)

# Estimate PET using temperature-based equation
data['PET'] = 0.0053 * (data['T'] + 17.8) * np.sqrt(data['T'].max() - data['T'].min()) * 25

## Run Simulations
Run the hydrological model with a range of `α` values that control system dynamics.  
Each simulation will be analyzed for causality.

In [ ]:
alphas = [1e-8, 0.5, 1, 2, 5]
simulations = [model(data, alpha=alpha).iloc[365*6:] for alpha in alphas]

## Run Sensitivity/Causality Analyses
Apply each causality or sensitivity method to the simulated data, using the best lag between precipitation and runoff.

In [ ]:
source_vars = ['S', 'ET', 'Qbase']
target_var = 'Qsim'
methods = ['CCM', 'Granger', 'MI', 'TE', 'SHAP']
results = []

for sim in tqdm(simulations, desc="Processing simulations"):
    best_lag = lagselector.LagSelector(sim, precip_col='P', runoff_col=target_var, max_lag=30).get_best_lag()

    # Run all methods
    result = [
        gc.granger_causality(sim, source_vars, target_var),
        te.transfer_entropy(sim, source_vars, target_var, best_lag, bins=10),
        mi.mutualinformation(sim, source_vars, target_var, best_lag, bins=10),
        ccm.ccmCausality(sim, source_vars, target_var, best_lag),
        sb.SHAPsensitivity(sim, source_vars, target_var, best_lag)
    ]
    results.append(result)

## Extract Summary Metrics
Aggregate results from each method and simulation.  
This step prepares the data for visualization and comparison across variables and methods.

In [ ]:
allsummaries = [[res.summary for res in res_list] for res_list in results]

# Extract metrics and variable names
metrics = [res.metric for res in results[0]]
methods = [res.method for res in results[0]]
variables = allsummaries[0][0].reset_index().rename(columns={0: 'Variables'})['Variables'].tolist()

# Initialize data container
metric_data = {metric: {var: [] for var in variables} for metric in metrics}

# Fill metric_data
for res_per_alpha in allsummaries:
    for res_df, metric in zip(res_per_alpha, metrics):
        df = res_df.reset_index() if 'Variables' not in res_df.columns else res_df
        df.rename(columns={df.columns[0]: 'Variables'}, inplace=True)
        for var in variables:
            val = df.loc[df['Variables'] == var, metric].values
            metric_data[metric][var].append(val[0] if len(val) else np.nan)

## Plot Results
Visualize the relationship between `α` values and the causality/sensitivity scores from each method.  
This helps identify which variables are most influential under varying dynamics.


In [ ]:
fig = plt.figure(figsize=(12, 6))
plt.subplots_adjust(right=0.75, hspace=0.4, wspace=0.3)
colors = cm.get_cmap('tab10', len(variables))

for i, metric in enumerate(metrics):
    ax = plt.subplot(2, 3, i + 1)
    ax.axvline(x=0.5, lw=5, color='salmon', alpha=0.5)

    for j, var in enumerate(variables):
        ax.plot(alphas, metric_data[metric][var], marker='o', label=var,
                color=colors(j), markeredgecolor='k')

    ax.set_title(methods[i], fontsize=14)
    ax.set_ylabel(metric)
    ax.set_xlabel('α')
    ax.set_xticks(alphas)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legend outside the plot
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles, labels,
    title='Variables',
    loc='center right',
    bbox_to_anchor=(0.9, 0.3),
    frameon=True,
    fancybox=True,
    shadow=True,
    framealpha=0.9,
    edgecolor='black',
    facecolor='whitesmoke'
)

plt.tight_layout()
plt.show()